<a href="https://colab.research.google.com/github/Nilufar-Komil/Applications-of-the-Generalized-Stiefel-Manifold-in-NN/blob/main/ViT_sgd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# -*- coding: utf-8 -*-

import os
import sys
import math
import random
from typing import Optional

sys.path.append(os.getcwd())

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.optim import SGD
from torch.utils.data import Dataset, DataLoader, random_split

# Data processing and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device available:", device)


Device available: cpu


In [2]:

# ------------------------------------------------------------
# CIFAR-10 normalization (same as WRN-28-10)
# ------------------------------------------------------------

cifar10_mean = (125.3 / 255.0, 123.0 / 255.0, 113.9 / 255.0)
cifar10_std  = (63.0 / 255.0, 62.1 / 255.0, 66.7 / 255.0)

# ------------------------------------------------------------
# Training transform
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Pad(4, padding_mode='reflect'),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# ------------------------------------------------------------
# Validation and test transform
# ------------------------------------------------------------

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# ------------------------------------------------------------
# Load CIFAR-10 datasets
# ------------------------------------------------------------

full_train_dataset_aug = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

full_train_dataset_clean = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=test_transform,
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

# ------------------------------------------------------------
# Train / validation split (80% / 20%)
# Same indices as WRN-28-10
# ------------------------------------------------------------

train_size = int(0.8 * len(full_train_dataset_aug))
val_size = len(full_train_dataset_aug) - train_size
split_seed = 42

train_dataset, _ = random_split(
    full_train_dataset_aug,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(split_seed),
)

_, val_dataset = random_split(
    full_train_dataset_clean,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(split_seed),
)

print("Full train size:", len(full_train_dataset_aug))
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)


100%|██████████| 170M/170M [05:27<00:00, 521kB/s]


Full train size: 50000
Train size: 40000
Val size: 10000
Test size: 10000


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [3]:

# ------------------------------------------------------------
# Multi-Head Self-Attention
# ------------------------------------------------------------

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3,
                 dropout=0.0):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Separate per-head Q, K, V matrices
        # Shape: (num_heads, embed_dim, head_dim)

        shape = (num_heads, embed_dim, self.head_dim)

        self.W_q = nn.Parameter(torch.empty(shape))
        self.W_k = nn.Parameter(torch.empty(shape))
        self.W_v = nn.Parameter(torch.empty(shape))

        self.b_q = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )
        self.b_k = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )
        self.b_v = nn.Parameter(
            torch.zeros(num_heads, self.head_dim)
        )

        self.proj = nn.Linear(embed_dim, embed_dim)

        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):
        # Orthogonal initialization, as in WRN-SGD
        for h in range(self.num_heads):
            nn.init.orthogonal_(self.W_q[h])
            nn.init.orthogonal_(self.W_k[h])
            nn.init.orthogonal_(self.W_v[h])

        nn.init.zeros_(self.b_q)
        nn.init.zeros_(self.b_k)
        nn.init.zeros_(self.b_v)

        nn.init.orthogonal_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def project(self, x, W, bias):
        # x: (batch, tokens, embed_dim)
        # W: (heads, embed_dim, head_dim)

        out = torch.einsum("bnd,hdp->bhnp", x, W)

        return out + bias[None, :, None, :]

    def forward(self, x):
        B, N, C = x.shape

        q = self.project(x, self.W_q, self.b_q)
        k = self.project(x, self.W_k, self.b_k)
        v = self.project(x, self.W_v, self.b_v)

        # Attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale

        attn = torch.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values
        out = attn @ v

        # Combine attention heads
        out = out.transpose(1, 2).reshape(B, N, C)

        out = self.proj(out)
        out = self.proj_drop(out)

        return out


# ------------------------------------------------------------
# Transformer MLP
# ------------------------------------------------------------

class TransformerMLP(nn.Module):
    def __init__(self, embed_dim=192, mlp_ratio=4.0,
                 dropout=0.0):
        super().__init__()

        hidden_dim = int(embed_dim * mlp_ratio)

        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.drop2 = nn.Dropout(dropout)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.orthogonal_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)

        nn.init.orthogonal_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.drop2(x)

        return x


# ------------------------------------------------------------
# ViT Transformer Block
# Replaces WRNBasicBlock
# ------------------------------------------------------------

class ViTBasicBlock(nn.Module):
    def __init__(self, embed_dim=192, num_heads=3,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attn = MultiHeadSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = TransformerMLP(
            embed_dim=embed_dim,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.ones_(self.norm1.weight)
        nn.init.zeros_(self.norm1.bias)

        nn.init.ones_(self.norm2.weight)
        nn.init.zeros_(self.norm2.bias)

    def forward(self, x):
        # Multi-head attention + residual
        x = x + self.attn(self.norm1(x))

        # MLP + residual
        x = x + self.mlp(self.norm2(x))

        return x


In [4]:

import torch
import torch.nn as nn


# ------------------------------------------------------------
# ViT Group: replaces WRNGroup
# ------------------------------------------------------------

class ViTGroup(nn.Module):
    def __init__(self, num_blocks, embed_dim=192,
                 num_heads=3, mlp_ratio=4.0, dropout=0.0):
        super().__init__()

        blocks = []

        for i in range(num_blocks):
            blocks.append(
                ViTBasicBlock(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
            )

        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


# ------------------------------------------------------------
# ViT-Tiny/4: replaces WideResNet
# ------------------------------------------------------------

class ViTTiny4(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        depth=12,
        embed_dim=192,
        num_heads=3,
        num_classes=10,
        mlp_ratio=4.0,
        dropout=0.0,
    ):
        super().__init__()

        assert img_size % patch_size == 0
        assert depth % 3 == 0
        assert embed_dim % num_heads == 0

        self.img_size = img_size
        self.patch_size = patch_size

        self.num_patches = (img_size // patch_size) ** 2

        # ----------------------------------------------------
        # Patch embedding
        # 3 x 32 x 32 -> 192 x 8 x 8
        # ----------------------------------------------------

        self.patch_embed = nn.Conv2d(
            in_channels=3,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=True,
        )

        # Classification token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )

        # Learnable position embeddings
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches + 1, embed_dim)
        )

        self.pos_drop = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Transformer encoder groups
        # 12 blocks = 3 groups x 4 blocks
        # ----------------------------------------------------

        blocks_per_group = depth // 3

        self.group0 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.group1 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.group2 = ViTGroup(
            num_blocks=blocks_per_group,
            embed_dim=embed_dim,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        self.reset_parameters()

    # --------------------------------------------------------
    # Initialization
    # --------------------------------------------------------

    def reset_parameters(self):
        nn.init.orthogonal_(self.patch_embed.weight)
        nn.init.zeros_(self.patch_embed.bias)

        nn.init.normal_(
            self.cls_token, mean=0.0, std=0.02
        )

        nn.init.normal_(
            self.pos_embed, mean=0.0, std=0.02
        )

        nn.init.ones_(self.norm.weight)
        nn.init.zeros_(self.norm.bias)

        nn.init.normal_(
            self.head.weight,
            mean=0.0,
            std=2 / (self.head.in_features ** 0.5)
        )
        nn.init.zeros_(self.head.bias)

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    def forward(self, x):

        # Patch embedding
        # (B, 3, 32, 32) -> (B, 192, 8, 8)
        x = self.patch_embed(x)

        # Flatten spatial dimensions
        # (B, 192, 8, 8) -> (B, 64, 192)
        x = x.flatten(2).transpose(1, 2)

        # Add classification token
        cls_token = self.cls_token.expand(
            x.shape[0], -1, -1
        )

        x = torch.cat((cls_token, x), dim=1)

        # Add positional embeddings
        # (B, 65, 192)
        x = self.pos_drop(x + self.pos_embed)

        # Transformer groups
        x = self.group0(x)
        x = self.group1(x)
        x = self.group2(x)

        # Final LayerNorm
        x = self.norm(x)

        # Extract CLS token
        x = x[:, 0]

        # Classification
        x = self.head(x)

        return x


In [5]:

model = ViTTiny4(
    img_size=32,
    patch_size=4,
    depth=12,
    embed_dim=192,
    num_heads=3,
    num_classes=10,
    mlp_ratio=4.0,
    dropout=0.0,
).to(device)

print("Total parameters:",
      sum(p.numel() for p in model.parameters()))

# Forward pass test
x = torch.randn(2, 3, 32, 32).to(device)
y = model(x)

print("Output shape:", y.shape)


Total parameters: 5362762
Output shape: torch.Size([2, 10])


In [6]:

# ------------------------------------------------------------
# Initialize ViT-Tiny/4
# ------------------------------------------------------------

model = ViTTiny4(
    img_size=32,
    patch_size=4,
    depth=12,
    embed_dim=192,
    num_heads=3,
    num_classes=10,
    mlp_ratio=4.0,
    dropout=0.0,
).to(device)

# ------------------------------------------------------------
# Loss function
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------
# SGD optimizer
# ------------------------------------------------------------

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
)


In [7]:

# ------------------------------------------------------------
# Learning-rate scheduler settings
# ------------------------------------------------------------

LR_MILESTONES = [12, 60, 120, 160]
LR_GAMMA = 0.2

euclid_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

# ------------------------------------------------------------
# Number of training epochs
# ------------------------------------------------------------

num_epochs = 200

print("Starting ViT-Tiny/4 training...")
print("Device:", device)
print("Number of epochs:", num_epochs)
print("Batch size:", batch_size)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])


Starting ViT-Tiny/4 training...
Device: cpu
Number of epochs: 200
Batch size: 128
Initial learning rate: 0.1


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4: Training and validation
# ------------------------------------------------------------

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):

    # -----------------------------
    # Training
    # -----------------------------
    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # ViT-Tiny/4 forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_train_loss = running_train_loss / train_total
    epoch_train_acc = train_correct / train_total

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # -----------------------------
    # Validation
    # -----------------------------
    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / val_total
    epoch_val_acc = val_correct / val_total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    # -----------------------------
    # Learning-rate scheduler
    # -----------------------------

    # Learning rate used during this epoch
    current_lr_euclid = optimizer.param_groups[0]["lr"]

    # Update scheduler for the next epoch
    euclid_scheduler.step()

    # -----------------------------
    # Print results
    # -----------------------------

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Train Acc: {epoch_train_acc:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val Acc: {epoch_val_acc:.4f} | "
        f"LR: {current_lr_euclid:.6f}"
    )

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:

# ------------------------------------------------------------
# ViT-Tiny/4 SGD: Test evaluation
# ------------------------------------------------------------

model.eval()

test_running_loss = 0.0
test_running_correct = 0
test_running_total = 0

# CIFAR-10 class names
classes = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

# Per-class statistics
class_correct = {cls_name: 0 for cls_name in classes}
class_total = {cls_name: 0 for cls_name in classes}

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # ViT-Tiny/4 forward pass
        outputs = model(images)

        # Test loss
        loss = criterion(outputs, labels)

        test_running_loss += loss.item() * images.size(0)

        # Predictions
        _, predicted = torch.max(outputs, dim=1)

        test_running_correct += (
            predicted == labels
        ).sum().item()

        test_running_total += labels.size(0)

        # Per-class accuracy
        for true_label, pred_label in zip(labels, predicted):

            true_idx = true_label.item()
            cls_name = classes[true_idx]

            class_total[cls_name] += 1

            if pred_label.item() == true_idx:
                class_correct[cls_name] += 1

# ------------------------------------------------------------
# Overall test metrics
# ------------------------------------------------------------

test_loss = test_running_loss / test_running_total
test_accuracy = test_running_correct / test_running_total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# ------------------------------------------------------------
# Per-class accuracy
# ------------------------------------------------------------

print("\nPer-class accuracy:")

class_accuracies = {}

for cls_name in classes:

    if class_total[cls_name] > 0:

        acc = class_correct[cls_name] / class_total[cls_name]

        class_accuracies[cls_name] = acc

        print(f"{cls_name:>5s}: {acc:.4f}")


# ------------------------------------------------------------
# Save training metrics to CSV
# ------------------------------------------------------------

output_dir = "vit_sgd"
os.makedirs(output_dir, exist_ok=True)

results = pd.DataFrame({
    "epoch": list(range(1, len(train_losses) + 1)),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "val_loss": val_losses,
    "val_accuracy": val_accuracies,
})

metrics_path = os.path.join(
    output_dir, "training_metrics.csv"
)

results.to_csv(metrics_path, index=False)


# ------------------------------------------------------------
# Save ViT-Tiny/4 SGD checkpoint
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    output_dir, "vit_tiny4_sgd_checkpoint.pt"
)

torch.save(
    {
        # Architecture configuration
        "model_name": "vit_tiny4",
        "dataset": "CIFAR10",
        "img_size": 32,
        "patch_size": 4,
        "depth": 12,
        "embed_dim": 192,
        "num_heads": 3,
        "num_classes": 10,
        "mlp_ratio": 4.0,
        "dropout": 0.0,

        # Training settings
        "num_epochs": num_epochs,
        "batch_size": batch_size,
        "lr_milestones": LR_MILESTONES,
        "lr_gamma": LR_GAMMA,

        # Model and optimizer states
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": euclid_scheduler.state_dict(),

        # Training history
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,

        # Test results
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "class_accuracies": class_accuracies,
    },
    checkpoint_path,
)

print(f"Saved {metrics_path}")
print(f"Saved {checkpoint_path}")


In [8]:

# ------------------------------------------------------------
# ViT-Tiny/4 SGD: Training plots
# ------------------------------------------------------------

import matplotlib.pyplot as plt
import os

output_dir = "vit_sgd"
os.makedirs(output_dir, exist_ok=True)

epochs = range(1, len(train_losses) + 1)

# ------------------------------------------------------------
# Training and validation loss
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs, train_losses,
    marker='o', label='Train Loss'
)

plt.plot(
    epochs, val_losses,
    marker='s', label='Validation Loss'
)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ViT-Tiny/4 SGD: Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(output_dir, "loss_curves.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------
# Training and validation accuracy (%)
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    [x * 100 for x in train_accuracies],
    marker='o',
    label='Train Accuracy'
)

plt.plot(
    epochs,
    [x * 100 for x in val_accuracies],
    marker='s',
    label='Validation Accuracy'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('ViT-Tiny/4 SGD: Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.savefig(
    os.path.join(output_dir, "accuracy_curves.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved:", os.path.join(output_dir, "loss_curves.png"))
print("Saved:", os.path.join(output_dir, "accuracy_curves.png"))



NameError: name 'train_losses' is not defined